# Graphql

> Fill in a module description here

In [ ]:
# | default_exp graphql

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [ ]:
# | export
from typing import (
    Annotated,
    Any,
    Union,
    cast,
    Sequence,
    Callable,
    TypeVar,
    TypedDict,
    List,
    Optional,
)
from functools import wraps
from enum import Enum
from uuid import UUID
import strawberry
import jwt
from random import randint
from jwt.exceptions import InvalidTokenError
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from strawberry.extensions import AddValidationRules
from graphql.validation import NoSchemaIntrospectionCustomRule
from product_horse.filesystems import R2StorageClient, FilePathType, LocalFileSystem
from product_horse.db import (
    SqlModelDatabase,
    Employee,
    PermissionLevel as DbPermissionLevel,
    Company,
    UnvalidatedCompany,
    CreateEmployee,
    User as DbUser,
    Utterance,
    UnvalidatedUser,
    Word,
    Video,
    Clip,
    FileMetadata,
    UnvalidatedFileMetadata,
    Transcription,
    UtteranceSegment,
    UnvalidatedTranscription,
    UnvalidatedUtterance,
    FileType as DbFileType,
)
from datetime import datetime, timedelta

from psycopg2 import OperationalError

from pydantic import ValidationError
from product_horse.audio_video import AudioEditor, create_video_from_utterances as create_video
from product_horse.api import get_relevant_utterances_from_query

from strawberry.permission import BasePermission
from strawberry.file_uploads import Upload
from strawberry.fastapi import BaseContext, GraphQLRouter
from starlette.datastructures import UploadFile
from tenacity import retry, stop_after_attempt, wait_exponential

from dotenv import load_dotenv
import os

if not os.getenv("DATABASE_URL"):
    load_dotenv()

True

# Auth and context

In [ ]:
# |export
secret = os.getenv("SECRET")
API_URL = "https://storage.producthorse.workers.dev/"  # or localhost:8787 - but won't work with transcription
# API_URL = "http://localhost:8787"

In [ ]:
# | export
database_url = os.getenv("DATABASE_URL")
database_superuser_url = os.getenv("DATABASE_SUPERUSER_URL")
if database_url is None or database_superuser_url is None:
    raise ValueError("DATABASE_URL or DATABASE_SUPERUSER_URL is not set")
if secret is None:
    raise ValueError("SECRET is not set")
long_timeout = "connect_timeout=10&keepalives=1&keepalives_idle=5&keepalives_interval=2&keepalives_count=2"
database = SqlModelDatabase(
    database_url=database_url + "&" + long_timeout
)
database_superuser = SqlModelDatabase(
    database_url=database_superuser_url
)
api_key = os.getenv("ASSEMBLYAI_API_KEY")
if not api_key:
    raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
audio_editor = AudioEditor(api_key=api_key)


class JwtPayload(TypedDict):
    id: UUID
    company_id: UUID
    exp: float
    permission_level: DbPermissionLevel


def create_jwt(
    employee_id: UUID, company_id: UUID, permission_level: DbPermissionLevel
) -> str:
    expiration = datetime.utcnow() + timedelta(weeks=2)
    payload = {
        "id": str(employee_id),
        "company_id": str(company_id),
        "exp": expiration,
        "permission_level": permission_level.value,
    }
    return jwt.encode(payload, secret, algorithm="HS256")  # type: ignore


def decode_jwt(token: str) -> JwtPayload:
    payload = cast(JwtPayload, jwt.decode(token, secret, algorithms=["HS256"]))  # type: ignore
    if "id" not in payload:
        raise InvalidTokenError
    if "company_id" not in payload:
        raise InvalidTokenError
    if datetime.fromtimestamp(payload["exp"]) < datetime.utcnow():
        raise InvalidTokenError
    return payload


class Context(BaseContext):
    @property
    def employee(self) -> Optional[Employee]:
        request = self.request
        if request is None:
            print("Request is None")
            return None
        authorization = request.headers.get("Authorization")
        if authorization is None:
            return None
        if not authorization.startswith("Bearer "):
            return None
        token = authorization.split()[1]
        payload = decode_jwt(token)
        employee_id = payload["id"]
        # run on superuser connection, since it's behind jwt auth
        employee = database_superuser.get_employee(employee_id=employee_id)
        return employee

    @property
    def jwt(self) -> Optional[str]:
        request = self.request
        if request is None:
            return None
        authorization = request.headers.get("Authorization")
        if authorization is None:
            return None
        if not authorization.startswith("Bearer "):
            return None
        token = authorization.split()[1]
        return token


class IsAuthenticated(BasePermission):
    message = "Employee is not authenticated"
    error_extensions = {"code": "UNAUTHORIZED"}

    def has_permission(self, source: Any, info: strawberry.Info, **kwargs: Any) -> bool:
        employee = info.context.employee
        return employee is not None


T = TypeVar("T")


@strawberry.type
class FormError:
    field: str
    message: str


@strawberry.type
class FormErrors:
    errors: list[FormError]


def handle_form_validation_errors(
    func: Callable[..., T],
) -> Callable[..., Union[T, FormErrors]]:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Union[T, FormErrors]:
        try:
            return func(*args, **kwargs)
        except ValidationError as e:
            errors = [
                FormError(field=cast(str, error["loc"][-1]), message=error["msg"])
                for error in e.errors()
            ]
            return FormErrors(errors=errors)

    return wrapper

# Helpers

In [ ]:
# | export
# Something is wrong with this retry -- doesn't help, just adds latency

def retry_on_operational_error(func: Callable[..., T]) -> Callable[..., T]:
    @wraps(func)
    @retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=4, max=15))
    def wrapper(*args: Any, **kwargs: Any) -> T:
        try:
            return func(*args, **kwargs)
        except OperationalError:
            raise Exception("Database access error, try again in a minute")
    return wrapper

@retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=4, max=15))
async def transcribe_and_save_file(
    file_metadata: FileMetadata,
    signed_url: str,
    employee: Employee,
    db: SqlModelDatabase,
) -> Transcription:
    transcription_result = await audio_editor.transcribe(signed_url, employee)
    utterances: List[UnvalidatedUtterance] = transcription_result.utterances

    unvalidated_transcription = UnvalidatedTranscription(
        file_id=file_metadata.id,
        text=transcription_result.text,
        company_id=employee.company_id,
        created_by_id=employee.id,
    )
    transcription: Transcription = db.as_employee(employee).save_transcription(
        unvalidated_transcription, utterances
    )
    return transcription

# Graphql Types

In [ ]:
# | export
PermissionsLevel = strawberry.enum(DbPermissionLevel)


@strawberry.experimental.pydantic.type(Employee)
class Employee:
    id: UUID
    email: str
    name: str
    permission_level: PermissionsLevel


@strawberry.experimental.pydantic.type(DbUser)
class User:
    id: UUID
    name: str


@strawberry.experimental.pydantic.type(Word)
class Word:
    id: UUID
    text: str
    start: float
    end: float


@strawberry.input
class UtteranceSegmentInput:
    utterance_id: str
    word_ids: list[str]


@strawberry.experimental.pydantic.type(Utterance)
class Utterance:
    id: UUID
    confidence: float
    start_time: float
    end_time: float
    speaker: str
    text: str
    words: list[Word]


@strawberry.experimental.pydantic.type(Clip)
class Clip:
    id: UUID
    name: str
    words: list[Word]


@strawberry.experimental.pydantic.type(Video)
class Video:
    id: UUID
    title: str
    clips: list[Clip]
    title: str
    file_path: str
    resolution_x: int
    resolution_y: int
    render_status: str
    fps: int
    signed_url: Optional[str] = None
    visibility: str


@strawberry.experimental.pydantic.type(Company)
class Company:
    id: UUID
    name: str
    employees: list[Employee]


@strawberry.experimental.pydantic.type(FileMetadata)
class FileMetadata:
    id: UUID
    file_name: str
    file_path: str


@strawberry.experimental.pydantic.type(Transcription)
class Transcription:
    id: UUID
    text: str
    created_at: datetime
    updated_at: datetime
    embedded: Optional[bool]
    utterances: list[Utterance]
    file_metadata: FileMetadata


@strawberry.type
class RegisterCompanySuccess:
    company: Company
    token: str


@strawberry.enum
class FileType(Enum):
    VIDEO = "video"
    AUDIO = "audio"
    TEXT = "text"


@strawberry.input
class FileMetadataInput:
    file_name: Optional[str]
    file_type: FileType
    resolution_x: int
    resolution_y: int
    frame_rate: int
    duration: float


# Create a Union type to represent the 2 results from the mutation
RegisterResponse = Annotated[
    Union[RegisterCompanySuccess, FormErrors],
    strawberry.union("RegisterCompanyResponse"),
]


@strawberry.type
class LoginSuccess:
    employee: Employee
    token: str


LoginResponse = Annotated[
    Union[LoginSuccess, FormErrors],
    strawberry.union("LoginResponse"),
]

# Queries

In [ ]:
# | export
@strawberry.type
class Query:
    employee: Employee

    @strawberry.field(permission_classes=[IsAuthenticated])
    def get_users(self, info: strawberry.Info) -> Sequence[User]:
        return database.as_employee(info.context.employee).get_all_users()  # type: ignore

    @strawberry.field(permission_classes=[IsAuthenticated])
    @retry_on_operational_error
    def get_transcripts(self, info: strawberry.Info) -> Sequence[Transcription]:
        transcripts = database.as_employee(
            info.context.employee
        ).get_all_unique_transcriptions(mode="file_name")
        return transcripts  # type: ignore

    @strawberry.field(permission_classes=[IsAuthenticated])
    @retry_on_operational_error
    async def get_relevant_utterances(
        self, info: strawberry.Info, query: str, transcript_ids: Sequence[str]
    ) -> Sequence[Utterance]:
        transcript_uuids = [UUID(transcript_id) for transcript_id in transcript_ids]
        transcripts = database.as_employee(info.context.employee).get_transcriptions(
            transcription_ids=transcript_uuids
        )
        # TODO: this won't work if there are too few utterances to query. Add a check.
        # TODO: check the seconds buffer and N to return in the search_engine. Might be worth exposing those,
        # and I am not sure what seconds buffer is anyway.
        utterances = await get_relevant_utterances_from_query(
            db=database,
            employee=info.context.employee,
            query=query,
            transcripts=transcripts,
        )
        return utterances  # type: ignore

    @strawberry.field(permission_classes=[IsAuthenticated])
    @retry_on_operational_error
    def get_video(self, info: strawberry.Info, video_id: str) -> Video:
        video = database.as_employee(info.context.employee).get_video(video_id)
        if video is None:
            raise Exception("Video not found")
        
        r2 = R2StorageClient(
            api_url=API_URL,
            base_path=info.context.employee.company_id,
            jwt=info.context.jwt,
        )
        if video.file_path is None:
            raise Exception("Video file path is None")
        signed_url = r2.get_signed_url(video.file_path)
        video_dict = video.model_dump()
        # Remove unwanted keys from the dictionary
        keys_to_remove = ["created_by_id", "company_id", "updated_at", "created_at"]
        for key in keys_to_remove:
            video_dict.pop(key, None)
        video_dict["signed_url"] = signed_url
        try:
            if video_dict["clips"] is None:
                video_dict["clips"] = video.clips or []
        except KeyError as e:
            print(e)
            video_dict["clips"] = []
        return Video(**video_dict)

    @strawberry.field(permission_classes=[IsAuthenticated])
    @retry_on_operational_error
    def get_all_videos(self, info: strawberry.Info) -> Sequence[Video]:
        return database.as_employee(info.context.employee).get_all_videos()  # type: ignore

# Mutations

In [ ]:
# | export
@strawberry.type
class Mutation:
    @strawberry.mutation
    def login(self, email: str, password: str) -> LoginResponse:
        employee = database_superuser.authenticate_employee(email, password)
        if employee is None:
            raise Exception("Invalid email or password")
        return LoginSuccess(
            employee=employee,  # type: ignore
            token=create_jwt(
                employee.id, employee.company_id, employee.permission_level
            ),
        )

    @strawberry.mutation
    @retry_on_operational_error
    @handle_form_validation_errors
    def register_company_and_employee(
        self, name: str, email: str, password: str, company_name: str
    ) -> RegisterResponse:
        company_to_save = UnvalidatedCompany(name=company_name)
        employee_to_save = CreateEmployee(
            name=name,
            email=email,
            password=password,
            permission_level=DbPermissionLevel.ADMIN,
        )
        company, employee = database_superuser.save_company_and_employee(
            company_to_save, employee_to_save
        )
        return RegisterCompanySuccess(
            company=company,  # type: ignore
            token=create_jwt(employee.id, company.id, employee.permission_level),
        )

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    @retry_on_operational_error
    def save_user(
        self, info: strawberry.Info[Context, Any], name: str, external_id: Optional[str] = None
    ) -> User:
        external_id = external_id if external_id is not None else ""
        user = UnvalidatedUser(
            name=name,
            external_id=external_id,
            company_id=info.context.employee.company_id,
            created_by_id=info.context.employee.id,
        )
        return database.as_employee(info.context.employee).save_user(user)  # type: ignore

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    async def save_files_and_transcriptions(
        self,
        info: strawberry.Info,
        user_id: UUID,
        files: Sequence[Upload],
        file_metadata: FileMetadataInput,
    ) -> List[FileMetadata]:
        r2 = R2StorageClient(
            api_url=API_URL,
            base_path=info.context.employee.company_id,
            jwt=info.context.jwt,
        )
        """Takes a long time to respond because it's transcibing and saving each file.
        In the future could be restructured to save all files and then transcribe in the background.
        also need to save video filepath correctly. Right now its:
            filePath': '/var/folders/qj/d65y3xj97dj5gmv5gpb9bq3m0000gn/T/tmphx0pfwp0/0da7b2b4-03ca-4485-85d5-08955d5bf662/ad14cc47-331b-4573-bfd3-daa982df2c16.mp4
        
            V2: Objective: Avoid egress fees by providing client access to storage_client directly.
             - Faster upload because they can talk directly to the r2 api instead of the graphql server
             - handle 'file already exists' and pass storageserver errors to client

             Perhaps switch to Boto + Tigris since I am on Fly. This solves both issues as well.

            Design:
            1. two graphql endpoints for client: create_file_upload and query_file_upload.
                - Respond with status of the upload and the signed url if applicable
                - creates metadata and sets file status to "upload_initiated"
                - Handle the actual upload client-side, not in this server
            2. one graphql endpoint for server: file_upload_update
                -r2 api will send messages to this endpoint when the file upload is updated
                - r2 api will queue and send the transcription to this endpoint. This endpoint will create/save transcriptions.
            """
        user = database.as_employee(info.context.employee).get_user(user_id)
        if user is None:
            raise Exception("User not found")
        metadata: List[FileMetadata] = []
        for file in files:
            try:
                file = cast(UploadFile, file)
                derived_name = file.filename
                content_type = (
                    file.content_type if file.content_type is not None else "text/plain"
                )
                if derived_name is None and file_metadata.file_name is None:
                    raise Exception("File name is None")
                contents = await file.read()
                path = r2.build_user_path(user, FilePathType.USER_ID_BASE_TEXT)
                print("REMINDER to fix unreliable big uploads!!!!!!!!!!!!!!!!!!!!")
                print(path)
                final_name = derived_name if derived_name is not None else file_metadata.file_name
                file = await r2.create_file(
                    path, contents, final_name, authorized=True, mime_type=content_type
                )
                unvalidated_metadata = UnvalidatedFileMetadata(
                    user_id=user.id,
                    file_name=final_name,
                    file_path=file.uri,
                    created_by_id=info.context.employee.id,
                    company_id=info.context.employee.company_id,
                    file_type=DbFileType[file_metadata.file_type.value],
                    resolution_x=file_metadata.resolution_x,
                    resolution_y=file_metadata.resolution_y,
                    frame_rate=file_metadata.frame_rate,
                )
                file_metadata_to_save = database.as_employee(
                    info.context.employee
                ).save_file_metadata(unvalidated_metadata)
                metadata.append(
                    file_metadata_to_save  # type: ignore
                )
                # fix name issue
                signed_url = r2.get_signed_url(file_metadata_to_save.file_path)
                await transcribe_and_save_file(
                    file_metadata_to_save, signed_url, info.context.employee, database
                )
            except Exception as e:
                print(e)
                raise Exception(f"Failed to upload file {file}")
        return metadata

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    async def create_video_from_utterances(
        self,
        info: strawberry.Info,
        utterance_segments: Sequence[UtteranceSegmentInput],
        title: str,
    ) -> Video:
        """Refactor to be an initializer and then you query a different endpoint to get the video.
        Use render status appropriately.
        Mirror the v2 file upload design for this endpoint.
        1. create_video_from_utterances is an initializer, send id and respond with a video created
        - in background, kick off serverless video creation on modal.com. Use python storage_client to get the files there, not here.
        - modal will talk directly to the database and update the video with the status as it goes.
        2. get_video, responds with video and status, along with a signed url to download the video if it's ready.
        - signed url is for the r2 api, not the graphql server, so the client can download the video directly.
        """
        r2 = R2StorageClient(
            api_url=API_URL,
            base_path=info.context.employee.company_id,
            jwt=info.context.jwt,
        )
        server_file_system = LocalFileSystem()
        utterance_ids = [UUID(segment.utterance_id) for segment in utterance_segments]
        word_ids = [word_id for segment in utterance_segments for word_id in segment.word_ids]
        utterances = database.as_employee(info.context.employee).get_utterances(
            utterance_ids, word_ids=word_ids
        )
        # check to see if video title already exists
        video_exists = database.as_employee(info.context.employee).get_all_videos()
        for video in video_exists:
            if video.title == title:
                title = f"{title}-{randint(1, 200)}"
        utterance_segments_for_video : list[UtteranceSegment] = []
        for utterance in utterances:
            utterance_segments_for_video.append(
                UtteranceSegment(
                    id=utterance.id,
                    transcription_id=utterance.transcription_id,
                    transcription=utterance.transcription,
                    words=utterance.words,
                    segment_start_word=utterance.words[0],
                    segment_end_word=utterance.words[-1],
                    confidence=utterance.confidence,
                    company_id=utterance.company_id,
                    created_by_id=utterance.created_by_id,
                    start=utterance.start,
                    end=utterance.end,
                )
            )
        user = database.as_employee(info.context.employee).get_all_users()[0]
        if user is None:
            raise Exception("User not found")
        final_video_file_path = server_file_system.build_user_path(user, FilePathType.USER_ID_BASE_VIDEO)
        final_destination = f"{final_video_file_path}/{title}.mp4".strip('/') #type: ignore
        print(f"Final destination: {final_destination}")
        video = await create_video(
            database,
            remote_file_system=r2,
            local_file_system=server_file_system,
            employee=info.context.employee,
            user=user,
            utterance_segments=utterance_segments_for_video,
            final_destination=final_destination,
            title=title,
        )
        return video  # type: ignore

In [ ]:
# | export
async def get_context() -> Context:
    return Context()


schema = strawberry.Schema(Query, Mutation, scalar_overrides={UploadFile: Upload}, extensions=[
        AddValidationRules([NoSchemaIntrospectionCustomRule]),
    ],)

graphql_app = GraphQLRouter(
    schema,
    context_getter=get_context,
    graphql_ide=None,
)

app = FastAPI()
origins = ["*"
]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(graphql_app, prefix="/graphql")

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore